# 🔍 Validación: Antes y Después de la Limpieza

Este notebook te permite ver con tus propios ojos cómo cambian los datos
antes y después de aplicar cada función de limpieza.

**Cómo usarlo:**
1. Corre la celda de configuración (Celda 1)
2. Corre la celda de carga de datos (Celda 2)
3. Ejecuta la validación de la columna que te interese

---

## Celda 1 — Configuración y carga de datos

In [5]:
# Carga los datos crudos desde Bronze

storage_account = "stecommercedirty01"  # cambia por el tuyo
container = "bronze"

path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/customers/"
df_raw = spark.read.option("multiLine", True).json(path)

print(f"✅ Cargados {df_raw.count()} registros")
print(f"📋 Columnas: {df_raw.columns}")

NameError: name 'spark' is not defined

---
## 📝 Cómo ver ANTES y DESPUÉS de una columna

La técnica es simple:
1. Seleccionas la columna original
2. Aplicas la función de limpieza
3. Muestras AMBAS columnas juntas para comparar

Usamos `.select()` para ver solo las columnas que importan.

### 🔤 Validación: columna `name`

In [ ]:
# ─── ANTES y DESPUÉS de limpiar 'name' ───────────────────────────────────────
from transformacion.cleaning.customer_cleaner import clean_text_column

# Aplicamos la limpieza
df_name_check = clean_text_column(df_raw, "name")

# Mostramos ORIGINAL vs LIMPIO lado a lado
# Esto te permite ver exactamente qué cambió
df_name_check.select(
    "name",        # columna original (tal como vino de la API)
    "name_clean"   # columna después de la limpieza
).display()

print("\n🔎 Solo los que CAMBIARON (original != limpio):")
df_name_check.filter(
    df_name_check["name"] != df_name_check["name_clean"]
).select("name", "name_clean").display()

In [0]:
# ─── Detectar problemas: ¿Cuántos quedaron como UNKNOWN? ──────────────────────
from pyspark.sql import functions as F

total = df_name_check.count()
unknown_count = df_name_check.filter(F.col("name_clean") == "UNKNOWN").count()
changed_count = df_name_check.filter(F.col("name") != F.col("name_clean")).count()

print(f"📊 Total de registros: {total:,}")
print(f"✏️  Registros que cambiaron: {changed_count:,} ({changed_count/total*100:.1f}%)")
print(f"❓ Quedaron como UNKNOWN:  {unknown_count:,} ({unknown_count/total*100:.1f}%)")

# Ver cuáles quedaron como UNKNOWN para entender por qué
if unknown_count > 0:
    print("\n⚠️ Registros que quedaron UNKNOWN:")
    df_name_check.filter(F.col("name_clean") == "UNKNOWN") \
        .select("name", "name_clean") \
        .display()

---
### 📧 Validación: columna `email`

In [0]:
# ─── ANTES y DESPUÉS de limpiar 'email' ──────────────────────────────────────
from transformacion.cleaning.customer_cleaner import clean_email_column

df_email_check = clean_email_column(df_raw)

# Ver todos: original vs limpio
df_email_check.select("email", "email_clean").display()

# Ver solo los que quedaron como UNKNOWN (emails inválidos)
print("\n❌ Emails que NO pasaron validación:")
df_email_check.filter(
    F.col("email_clean") == "UNKNOWN"
).select("email", "email_clean").display()

---
### 🌍 Validación: columna `country`

In [0]:
# ─── ANTES y DESPUÉS de normalizar 'country' ─────────────────────────────────
from transformacion.cleaning.country_normalizer import normalize_country

df_country_check = normalize_country(df_raw, spark)

# Ver todos los valores únicos: original vs estandarizado
# .distinct() evita mostrar miles de filas repetidas
print("🌍 Valores únicos de country: original → limpio")
df_country_check.select("country", "country_clean") \
    .distinct() \
    .orderBy("country") \
    .display()

# Resumen de resultados
print("\n📊 Distribución de resultados:")
df_country_check.groupBy("country_clean") \
    .count() \
    .orderBy(F.col("count").desc()) \
    .display()

# Ver los que necesitan revisión manual
revisar = df_country_check.filter(F.col("country_clean") == "REVISAR_MANUAL")
if revisar.count() > 0:
    print("\n⚠️ Países que necesitan revisión manual:")
    revisar.select("country").distinct().display()

---
### 📅 Validación: columnas de fecha

In [0]:
# ─── ANTES y DESPUÉS de formatear fechas ─────────────────────────────────────
from transformacion.cleaning.customer_cleaner import clean_date_columns

DATE_COLS = ["created_at", "updated_at", "registration_date"]

# Guardamos los originales ANTES de limpiar
df_antes = df_raw.select(
    "id",
    *DATE_COLS  # * desempaqueta la lista como argumentos
).limit(20)

# Aplicamos la limpieza
df_fechas_clean = clean_date_columns(df_raw, DATE_COLS)

# Tomamos los mismos registros después de limpiar
df_despues = df_fechas_clean.select("id", *DATE_COLS).limit(20)

print("📅 ANTES (raw):")
df_antes.display()

print("\n📅 DESPUÉS (limpio):")
df_despues.display()

# Detectar fechas que NO pudieron parsearse (quedaron null)
for col_name in DATE_COLS:
    nulls_after = df_fechas_clean.filter(F.col(col_name).isNull()).count()
    if nulls_after > 0:
        print(f"\n⚠️ '{col_name}': {nulls_after} fechas no pudieron parsearse")
        df_fechas_clean.filter(F.col(col_name).isNull()) \
            .select("id", col_name) \
            .display()

---
## 🔄 Validación COMPLETA: pipeline entero

In [0]:
# ─── Correr el pipeline completo y ver el resultado final ─────────────────────
from transformacion.pipeline_transformacion import run_transformacion

dataframes = {"customers": df_raw}
dataframes_clean = run_transformacion(dataframes, spark)
df_final = dataframes_clean["customers"]

# Ver schema del resultado (tipos de datos después del pipeline)
print("📋 Schema FINAL (tipos de datos correctos):")
df_final.printSchema()

# Ver muestra del resultado
print("\n👀 Muestra del resultado limpio:")
df_final.display()

# Diagnóstico de nulos en el resultado
print("\n🚨 Nulos en el resultado final:")
df_final.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_final.columns
]).display()